<a href="https://colab.research.google.com/github/devharsh/cyberquest-camp/blob/main/Day2/Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 2 Notebook: The Math of Strong Passwords

Run a cell with **Shift + Enter**. Work through the Modules in order.

**Modules in this notebook:**
1. Module 1: Password keyspace math
2. Module 2: Crack-time estimate
3. Module 3: Password strength meter
4. Module 4: Hashing and why salt matters

The slides tell you when to run each Module. After each Module, go back to the slides.


## Module 1: Password keyspace math
Possible passwords = N to the power L, where N is the size of the character set and L is the length.


In [ ]:
for name, N in {"lowercase":26, "+digits":36, "+upper":62, "+symbols":95}.items():
    print(f"{name:12} length 8 -> {N**8:,} combinations")


## Module 2: Crack-time estimate
Time to try every combination = combinations / guesses-per-second. Attackers can try billions per second.


In [ ]:
def crack_years(N, L, gps=1_000_000_000):
    return (N ** L) / gps / (60*60*24*365)

for L in [6, 8, 12, 16]:
    print(L, "chars ->", round(crack_years(95, L), 4), "years")


## Module 3: Password strength meter (challenge)
Combine length and variety into a simple score.


In [ ]:
def meter(pw):
    s = (len(pw)>=12) + any(c.isdigit() for c in pw) + any(c.isupper() for c in pw) + any(not c.isalnum() for c in pw)
    return ["Weak","Weak","Okay","Strong","Strong"][s]

for pw in ["P@ss1!", "purple-kite-river-stone"]:
    print(pw, "->", meter(pw))


## Module 4: Hashing and why salt matters
A hash is a one-way fingerprint. A salt is random text added before hashing so identical passwords get different hashes.


In [ ]:
import hashlib
def h(x): return hashlib.sha256(x.encode()).hexdigest()[:24]
print("no salt, same password -> same hash:")
print(h("squad123"), h("squad123"))
print("with different salts -> different hashes:")
print(h("saltA:squad123"), h("saltB:squad123"))


## CyberQuest Python Exercises - Randomness & Strong Passwords
Why `random` is predictable, why `secrets` is safe, and how to build a strong password generator. Run each cell in order; complete the STUDENT EXERCISE cells yourself.

---
# MODULE 4: Pseudorandomness

Randomness is critical in cryptography. Weak randomness has caused real-world
catastrophic failures (Android Bitcoin wallet bug, Debian OpenSSL key generation flaw).

| Source | Predictable? | Use for crypto? |
|--------|-------------|----------------|
| `random` module | YES (with seed) | Never |
| `random.SystemRandom` | No (OS entropy) | Only if needed |
| `secrets` module | No (OS entropy) | Yes -- preferred |
| `os.urandom()` | No (OS entropy) | Yes |


In [ ]:
import random, secrets, string, os, time
print('Module 4 ready')


## 4.1 Python random Module: Basic Operations


In [ ]:
import random

# Basic operations
print('randint(1, 100):', random.randint(1, 100))
print('choice([a,b,c]):', random.choice(['alpha', 'beta', 'gamma']))
print('uniform(0, 1)  :', random.uniform(0, 1))

data = list(range(10))
random.shuffle(data)
print('After shuffle  :', data)


## 4.2 Seed Reproducibility Demo

When you set a seed, the sequence is **completely reproducible**.
This is useful for testing but **catastrophic** if used for keys or tokens.


In [ ]:
import random

print('Run 1 with seed 42:')
random.seed(42)
seq1 = [random.randint(0, 100) for _ in range(5)]
print(' ', seq1)

print('Run 2 with seed 42 (identical!):')
random.seed(42)
seq2 = [random.randint(0, 100) for _ in range(5)]
print(' ', seq2)

print(f'Sequences identical: {seq1 == seq2}')  # Expected: True

print('\nNo seed (different each run):')
random.seed(None)
print(' ', [random.randint(0, 100) for _ in range(5)])


## 4.3 /dev/random vs /dev/urandom

On Linux/macOS, the OS provides two random devices:

- `/dev/random`: blocks until enough entropy is gathered from hardware events.
  Very slow. Historically more conservative but now equivalent on modern kernels.
- `/dev/urandom`: non-blocking, uses a CSPRNG seeded from OS entropy.
  Fast and cryptographically secure for all practical purposes.

Python's `os.urandom()` calls `/dev/urandom` (or the Windows equivalent).


In [ ]:
import os

# os.urandom -- cryptographically secure, non-blocking
random_bytes = os.urandom(16)
print(f'os.urandom(16) hex: {random_bytes.hex()}')
print(f'os.urandom(16) int: {int.from_bytes(random_bytes, "big")}')

# Reading /dev/urandom directly (Unix only)
import platform
if platform.system() != 'Windows':
    with open('/dev/urandom', 'rb') as f:
        dev_bytes = f.read(8)
    print(f'/dev/urandom 8 bytes: {dev_bytes.hex()}')
else:
    print('Windows: /dev/urandom not available, os.urandom() is the equivalent')


## 4.4 SystemRandom: OS-Level Randomness via random API


In [ ]:
import random, time

sr = random.SystemRandom()   # Uses os.urandom() under the hood
print('SystemRandom values (non-reproducible):')
print([sr.randint(0, 999) for _ in range(5)])

# Demonstrate: seeding SystemRandom has NO EFFECT
sr.seed(42)   # silently ignored
run1 = [sr.randint(0, 100) for _ in range(5)]
sr.seed(42)
run2 = [sr.randint(0, 100) for _ in range(5)]
print(f'\nSeed ignored -- runs differ: {run1 != run2}')
print(f'Run 1: {run1}')
print(f'Run 2: {run2}')


## 4.5 Student Challenge: Secure Password Generator

Use the `secrets` module (not `random`) to build a password generator that:
- Produces passwords of at least 16 characters
- Guarantees at least one lowercase, uppercase, digit, and special character
- Is cryptographically secure


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Implement a secure random password generator
#
# Requirements:
#   - Use secrets module (NOT random)
#   - Length >= 16 (default)
#   - Must include: lowercase, uppercase, digit, special char
#   - Should reject passwords that don't satisfy all requirements
#     (keep regenerating until all checks pass)
#
# Expected output example: 'G@7kMpX!2nRvLs#8'
# ==========================================

import secrets, string

def generate_secure_password(length: int = 16) -> str:
    """
    Generate a cryptographically secure random password.
    Guarantees at least one char from each required character class.
    """
    # TODO: implement this function
    # Hint:
    #   alphabet = string.ascii_lowercase + string.ascii_uppercase + ...
    #   Use a while loop to keep generating until all requirements met
    #   Use secrets.choice(alphabet) for each character
    pass

# Test it:
for i in range(5):
    pwd = generate_secure_password()
    if pwd:  # remove 'if pwd' once implemented
        print(f'Password {i+1}: {pwd} (length: {len(pwd)})')
